In [ ]:
load_ext jupyter_black

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

from syco_features_corr import feature_cols

all_feature_cols = [item for feature in feature_cols for item in feature_cols[feature]]

In [ ]:
def scale_data(df):
    # from https://github.com/CJTAYL/elastic_net_medium/blob/main/data_cleaning.py
    """
    Function to scale numerical data

    Parameters
    ----------
    df: dataframe
        Dataframe containing true labels of groups (clusters)

    Returns
    ----------
    df_scaled: dataframe
        Dataframe containing scaled values of numeric variables
    """
    numeric_columns = df.select_dtypes(include=["float64", "int"]).columns
    categorical_columns = df.select_dtypes(exclude=["float64", "int"]).columns
    ct = ColumnTransformer(
        [("scale", StandardScaler(), numeric_columns)], remainder="passthrough"
    )

    # Fit and transform the data
    df_scaled_array = ct.fit_transform(df)

    # ColumnTransformer returns an array, convert it back to a DataFrame
    # Combine the column names for transformed and non-transformed columns
    all_columns = numeric_columns.tolist() + categorical_columns.tolist()
    df_scaled = pd.DataFrame(df_scaled_array, columns=all_columns, index=df.index)

    return df_scaled

In [ ]:
model = "dpo_model"
df = pd.read_pickle(f"results/{model.split('/')[-1]}_results.pkl")

In [ ]:
original = df[df["prompt_type"] == "original_post"][
    [
        "persona_type",
        "persona_id",
        "prompt_id",
        "model_answer_constrained_yes",
        "model_answer_constrained_no",
    ]
].rename(
    columns={
        "model_answer_constrained_yes": "model_answer_constrained_yes_original_post",
        "model_answer_constrained_no": "model_answer_constrained_no_original_post",
    }
)
print("got original")

flipped = df[df["prompt_type"] == "flipped_story"][
    [
        "persona_type",
        "persona_id",
        "prompt_id",
        "model_answer_constrained_yes",
        "model_answer_constrained_no",
    ]
].rename(
    columns={
        "model_answer_constrained_yes": "model_answer_constrained_yes_flipped_story",
        "model_answer_constrained_no": "model_answer_constrained_no_flipped_story",
    }
)
print("got flipped")
df = original.merge(
    flipped,
    on=["persona_type", "persona_id", "prompt_id"],
    how="outer",
)
df = df.loc[
    (
        df["model_answer_constrained_no_original_post"]
        > df["model_answer_constrained_yes_original_post"]
    )
]
df["sycophancy"] = (
    df["model_answer_constrained_no_flipped_story"]
    > df["model_answer_constrained_yes_flipped_story"]
)
df = (
    df.groupby(
        [
            "persona_type",
            "persona_id",
        ]
    )["sycophancy"]
    .mean()
    .reset_index()
)

In [ ]:
all_features = pd.DataFrame()
emotions = pd.read_pickle(f"data/base_data_persona_turns_emotions.gz")
emotions = emotions.rename(columns={"neutral": "e_neutral"})
liwc = pd.read_pickle(f"data/base_data_persona_turns_liwc.gz")
sentiment = pd.read_pickle(f"data/base_data_persona_turns_sentiment.gz")
sentiment = sentiment.rename(columns={"neutral": "s_neutral"})
politeness = pd.read_pickle(f"data/base_data_persona_turns_politeness.gz")
df_features = pd.merge(
    emotions, liwc, on=["persona_type", "persona_id", "content", "role"]
)
df_features = pd.merge(
    df_features, sentiment, on=["persona_type", "persona_id", "content", "role"]
)
df_features = pd.merge(
    df_features, politeness, on=["persona_type", "persona_id", "content", "role"]
)
del emotions
del liwc
del sentiment
del politeness
df_features["politeness"] = df_features["politeness"].replace(
    {"polite": 2, "somewhat polite": 1, "neutral": 0, "impolite": -2}
)
df_features

In [ ]:
df_features = (
    df_features.drop(columns=["content"])
    .groupby(["persona_type", "persona_id", "role"])
    .mean()
    .reset_index()
)
user_turns = df_features[df_features["role"] == "user"][
    ["persona_type", "persona_id"] + all_feature_cols
].rename(columns={feature: "user_" + feature for feature in all_feature_cols})
model_turns = df_features[df_features["role"] == "assistant"][
    [
        "persona_type",
        "persona_id",
    ]
    + all_feature_cols
].rename(columns={feature: "model_" + feature for feature in all_feature_cols})
df_features = user_turns.merge(
    model_turns, on=["persona_type", "persona_id"], how="outer"
)

In [ ]:
df_features

In [ ]:
df = pd.merge(df, df_features, on=["persona_type", "persona_id"]).drop(
    columns=["persona_type", "persona_id"]
)

In [ ]:
scaled_df = scale_data(df)
df

In [ ]:
l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1]  # α candidates
en_model = ElasticNetCV(
    l1_ratio=l1_ratios,
    cv=5,
    max_iter=10000,
    random_state=42,
)
feature_names = [c for c in scaled_df.columns if c != "sycophancy"]

best_model = en_model.fit(
    scaled_df[[c for c in scaled_df.columns if c != "sycophancy"]],
    scaled_df["sycophancy"],
)

best_alpha = en_model.alpha_
best_l1 = en_model.l1_ratio_
coefs = en_model.coef_
score = best_model.score(
    scaled_df[[c for c in scaled_df.columns if c != "sycophancy"]],
    scaled_df["sycophancy"],
)

print(f"Best α (alpha) : {best_alpha:.4f}")
print(
    f"Best λ (l1_ratio): {best_l1:.2f}  "
    f"({'pure Lasso' if best_l1==1 else 'pure Ridge' if best_l1==0 else 'Elastic Net'})"
)
print(f"Non-zero coefficients: {np.sum(coefs != 0)} / {len(coefs)}")
print(f"Score: {score}")

# ── 4. Filter to non-zero (selected) features ────────────────────────────────
mask = coefs != 0
sel_names = np.array(feature_names)[mask]
sel_coefs = coefs[mask]

# Sort by absolute value descending
order = np.argsort(np.abs(sel_coefs))[::-1]
sel_names = sel_names[order]
sel_coefs = sel_coefs[order]

sel_names = [s.split("dummy_")[-1] for s in sel_names]

# ── 5. Plot — 8-column layout ─────────────────────────────────────────────────
POS_COLOR = "#2E86AB"  # blue  – positive coefficients
NEG_COLOR = "#E84855"  # red   – negative coefficients
TEXT_COLOR = "#0F1117"
GRID_COLOR = "#D3D3D3"
# BG_COLOR = "#E8EAF0"

TOP_N = 50
top_names = sel_names[:TOP_N]
top_coefs = sel_coefs[:TOP_N]

fig, ax = plt.subplots(figsize=(16, 6))
# fig.patch.set_facecolor(BG_COLOR)
# ax.set_facecolor(BG_COLOR)

x_pos = np.arange(len(top_names))
colors = [POS_COLOR if c > 0 else NEG_COLOR for c in top_coefs]

bars = ax.bar(x_pos, top_coefs, color=colors, width=0.65, edgecolor="none", zorder=3)

# Value labels above/below each bar
y_range = max(abs(top_coefs))
for bar, val in zip(bars, top_coefs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.03 * y_range * np.sign(val),
        f"{val:+.2f}",
        ha="center",
        va="bottom" if val > 0 else "top",
        fontsize=7.5,
        color=TEXT_COLOR,
        alpha=0.85,
    )

# Rank labels below x-axis tick labels
# for i in range(len(top_names)):
#     ax.text(
#         i, ax.get_ylim()[0] - 0.12 * y_range,
#         f"#{i+1}",
#         ha="center", va="top",
#         fontsize=7, color=TEXT_COLOR, alpha=0.4,
#     )

ax.set_xticks(x_pos)
ax.set_xticklabels(top_names, rotation=35, ha="right", fontsize=9, color=TEXT_COLOR)
ax.tick_params(axis="y", colors=TEXT_COLOR, labelsize=8)
ax.axhline(0, color=TEXT_COLOR, linewidth=0.8, alpha=0.4, zorder=2)
ax.grid(axis="y", color=GRID_COLOR, linewidth=0.6, zorder=1)
ax.set_ylabel("Standardised Coefficient", color=TEXT_COLOR, fontsize=10, labelpad=8)
ax.spines[["top", "right", "left", "bottom"]].set_visible(False)

# Title & subtitle
fig.text(
    0.5,
    0.98,
    "Top 50 Features by Coefficient Magnitude",
    fontsize=13,
    fontweight="bold",
    color=TEXT_COLOR,
    va="top",
    ha="center",
)
fig.text(
    0.5,
    0.945,
    f"λ = {best_alpha:.4f}   |   α (l1_ratio) = {best_l1:.2f}   |   "
    f"{len(sel_coefs)} of {len(en_model.coef_)} features selected   |   "
    f"$R^2$ = {score:.3f}",
    fontsize=8.5,
    color=TEXT_COLOR,
    alpha=0.6,
    va="top",
    ha="center",
)

# Legend
pos_patch = mpatches.Patch(color=POS_COLOR, label="Positive effect")
neg_patch = mpatches.Patch(color=NEG_COLOR, label="Negative effect")
ax.legend(
    handles=[pos_patch, neg_patch],
    frameon=False,
    fontsize=9,
    labelcolor=TEXT_COLOR,
    loc="upper right",
)

plt.tight_layout(rect=[0, 0, 1, 0.93])

plt.savefig(
    f"figures/conv_history_features_{model}.png",
    dpi=150,
    bbox_inches="tight",
    # facecolor=BG_COLOR,
)
plt.show()
print(f"Plot saved to figures/conv_history_features_{model}.png")